# What makes a house expensive?### Multivariate Linear Regression · from scratch · ≈ 2 hoursLast lecture you watched the matrix form of linear regression take shape on the board — one row of weights, one matrix of features, and the whole dataset's predictions tumble out in a single multiplication. Today you'll turn that math into working code and point it at a real dataset: **house prices in California**.Eight numbers describe each neighborhood — median income, number of rooms, how old the houses are, latitude, longitude, and so on. From those eight numbers, you'll predict the median house value. Your model will be a vector of eight weights that says, in effect: *"here is how much each thing matters."*### What you will build1. A `MultivariateLinearRegression` class with vectorized `fit` and `predict`2. A loss curve that proves gradient descent is doing its job3. **A learning-rate playground** — the single most important hyperparameter in all of ML4. A ranked feature-importance plot answering the question the title asks### Ground rules- Only **NumPy**, **Matplotlib**, and sklearn (for data loading and the final comparison)- No `sklearn.linear_model.LinearRegression` until the very last section- Keep your math vectorized. If you find yourself writing a `for` loop over training examples, stop and think about how a matrix multiply would do the same thing in one line.Let's build.

---## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

---## 2. The dataThe **California Housing** dataset: 20,640 California census blocks, each described by 8 features. The target is the **median house value** in that block, in units of $100,000.| Feature | What it is ||---|---|| `MedInc` | median income in the block (tens of thousands) || `HouseAge` | median house age (years) || `AveRooms` | average rooms per household || `AveBedrms` | average bedrooms per household || `Population` | block population || `AveOccup` | average household occupants || `Latitude` | geographic latitude || `Longitude` | geographic longitude |Before you train anything, go look at the data. Which features are on wildly different scales? Which ones do you expect to matter? Write down two guesses before you see the answer in Section 6.

In [ ]:
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

print(f"Features: {X.shape},   Target: {y.shape}")
print(f"Feature names: {feature_names}")
print(f"\nFirst 3 rows:\n{X[:3]}")
print(f"\nTarget range: ${y.min()*100:.0f}k  —  ${y.max()*100:.0f}k")

### 2.1 Why standardization is not optional hereLook at the scales. `MedInc` is around 3–8; `Population` is in the thousands; `Latitude` is 32–42. If you feed these to gradient descent as-is, the gradient for `Population` will dominate — not because population matters more, but because its numbers are bigger. The learning rate that works for one feature will overshoot for another.**Task:** Split the data 80/20 (random_state=42), then standardize the features. Fit the scaler on the training set only.

In [ ]:
# Split, standardize. Short.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled = scaler.transform(X_test)         # reuse train mean/std

print(f"Train: {X_train_scaled.shape},  Test: {X_test_scaled.shape}")
print(f"Train means (should all be ~0): {X_train_scaled.mean(axis=0).round(3)}")
print(f"Train stds  (should all be ~1): {X_train_scaled.std(axis=0).round(3)}")

---## 3. Build itHere is the math from the lecture, in one place.**Hypothesis** (for the whole training set at once):$$h_\theta(X) = X\theta + b$$**Cost** (half-mean-squared-error):$$J(\theta) = \frac{1}{2m}\sum_{i=1}^{m}\left(h_\theta(x^{(i)}) - y^{(i)}\right)^2$$**Gradient** (for every weight at once):$$\frac{\partial J}{\partial\theta} = \frac{1}{m}X^T(h_\theta(X) - y)$$**Update:** $\theta \leftarrow \theta - \alpha \cdot \nabla J$Your class should expose `fit`, `predict`, and keep a `loss_history`. Beyond that, how you structure it is up to you. Keep it vectorized — any `for` loop over training examples is a bug.

In [ ]:
class MultivariateLinearRegression:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.loss_history = []
        self.weights = None
        self.bias = 0.0

    def fit(self, X, y):
        # Initialize weights and bias, then run gradient descent.
        # Append the cost to self.loss_history every iteration.
        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0
        self.loss_history = []

        for _ in range(self.n_iterations):
            with np.errstate(all="ignore"):          # divergence may overflow; that's expected
                y_hat = X @ self.weights + self.bias
                error = y_hat - y
                cost = (error @ error) / (2 * m)      # J = 1/(2m) * sum(error^2)
                self.loss_history.append(cost)

                if not np.isfinite(cost):             # diverged: stop and keep what we have
                    break

                grad_w = (X.T @ error) / m            # (1/m) X^T (h - y)
                grad_b = error.sum() / m
                self.weights = self.weights - self.learning_rate * grad_w
                self.bias = self.bias - self.learning_rate * grad_b
        return self

    def predict(self, X):
        return X @ self.weights + self.bias

---## 4. First training runTrain with `learning_rate=0.1` and `n_iterations=1000`. Then plot the loss curve.

In [ ]:
# Train and plot the loss curve.
model = MultivariateLinearRegression(learning_rate=0.1, n_iterations=1000)
model.fit(X_train_scaled, y_train)

plt.plot(model.loss_history)
plt.xlabel("Iteration")
plt.ylabel("Cost J (half MSE)")
plt.title("Loss curve, learning rate = 0.1")
plt.show()

print(f"Initial loss: {model.loss_history[0]:.4f}")
print(f"Final loss:   {model.loss_history[-1]:.4f}")

Compute test-set **RMSE** (root mean squared error) in the original dollar units. Since `y` is in units of $100k, multiply by 100 to get an answer in thousands of dollars.> RMSE = √( mean( (y_true − y_pred)² ) )

In [ ]:
# RMSE on test set, in thousands of dollars.
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

y_pred = model.predict(X_test_scaled)
my_rmse = rmse(y_test, y_pred) * 100
print(f"Test RMSE: ${my_rmse:.1f}k")

---## 5. The learning-rate playgroundThis is the part where everything you've heard about gradient descent becomes real.Train your model with **six different learning rates**:```α ∈ { 0.001, 0.01, 0.1, 0.3, 1.0, 3.0 }```Keep `n_iterations=1000` for all of them. **Plot all six loss curves on one figure** (use a log scale on the y-axis — it makes very different losses comparable). Then write a short paragraph answering:1. Which α converged fastest?2. Which α converged to the lowest loss?3. What happens with α = 3.0? Why?4. What happens with α = 0.001? Why?If a run diverges (NaN or inf), that's not a bug in your code — it's the point of this exercise. Catch the NaN and plot what you have.

In [ ]:
learning_rates = [0.001, 0.01, 0.1, 0.3, 1.0, 3.0]

# Train one model per learning rate; collect each model's loss_history.
# Plot all curves on one figure with plt.yscale('log').
histories = {}
for lr in learning_rates:
    m_lr = MultivariateLinearRegression(learning_rate=lr, n_iterations=1000).fit(X_train_scaled, y_train)
    histories[lr] = m_lr.loss_history

plt.figure(figsize=(10, 6))
for lr, h in histories.items():
    h = np.array(h)
    finite = np.isfinite(h)
    plt.plot(np.arange(len(h))[finite], h[finite], label=f"alpha = {lr}")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost J (log scale)")
plt.title("Loss curves for six learning rates")
plt.legend()
plt.show()

for lr, h in histories.items():
    status = "DIVERGED" if not np.isfinite(h[-1]) else f"final loss {h[-1]:.4f}"
    print(f"alpha = {lr:<6}  iterations run: {len(h):<5}  {status}")

**Write your observations in this cell.**

1. **Fastest convergence:** α = 0.3 reaches the minimum in the fewest iterations of all the runs that converge (α = 0.1 is close behind).
2. **Lowest loss:** α = 0.1 and α = 0.3 both end at essentially the same, lowest loss, which is the least-squares minimum. α = 0.01 gets close but is still creeping down at iteration 1000.
3. **α = 3.0:** the loss explodes and hits inf/NaN within a few hundred iterations. The step is so large that each update jumps *past* the bottom of the bowl and lands higher up the other side. The next step overshoots even more, so the error grows exponentially instead of shrinking. α = 1.0 is also beyond the stable range for this data and blows up (or oscillates wildly), which shows the stable limit lies somewhere between 0.3 and 1.
4. **α = 0.001:** it is perfectly stable but far too timid. Each step is tiny, so after 1000 iterations the loss has barely dropped from where it started and is still falling. It would get there eventually, but it needs many more iterations.

**Take-away:** too small wastes compute, too large diverges, and the best α sits just below the edge of instability.

---## 6. What actually makes a house expensive?Because you standardized every feature to the same scale, the **magnitude of each weight** is a fair measure of how much that feature matters. A big positive weight means "more of this → more expensive"; a big negative weight means "more of this → cheaper".Using the model trained in Section 4 (α = 0.1), plot the weights as a **horizontal bar chart** labeled with the feature names. Sort them by absolute magnitude so the biggest drivers are easy to see. Color positive bars one color, negative bars another.Now go back to your two guesses from Section 2. Were you right?

In [ ]:
# Horizontal bar chart of learned weights, sorted by absolute magnitude.
w = model.weights
order = np.argsort(np.abs(w))              # ascending, so the biggest ends up on top
names = np.array(feature_names)[order]
vals = w[order]
colors = ["tab:green" if v > 0 else "tab:red" for v in vals]

plt.barh(names, vals, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Weight (effect of +1 std dev on price, in $100k)")
plt.title("What drives California house prices?")
plt.show()

for n_, v_ in zip(names[::-1], vals[::-1]):
    print(f"{n_:<12} {v_:+.3f}")

**Were my guesses right?** The biggest drivers are `MedInc`, `Latitude` and `Longitude`. Richer blocks are more expensive (positive `MedInc`), and the large negative `Latitude` and `Longitude` weights mean prices fall as you move north and east, away from the expensive coast (Bay Area and Los Angeles). So median income was the obvious guess, and location turned out to matter just as much, which is why combining the two coordinates into one feature (stretch goal A) should help.

A few weights deserve caution. `AveBedrms` is positive while `AveRooms` is negative, even though more rooms should intuitively mean a pricier house. These two features are highly correlated, so the model splits the credit between them in odd ways. `Population` and `AveOccup` have almost no effect once the other features are known.

---## 7. Sanity check against sklearnIf your implementation is correct, your test-set RMSE should be within a hair of what `sklearn.linear_model.LinearRegression` produces on the same data.

In [ ]:
from sklearn.linear_model import LinearRegression

# Train sklearn's model on the same scaled data, compare test RMSE side by side.
sk = LinearRegression().fit(X_train_scaled, y_train)
sk_rmse = rmse(y_test, sk.predict(X_test_scaled)) * 100

print(f"My model     RMSE: ${my_rmse:.2f}k")
print(f"sklearn      RMSE: ${sk_rmse:.2f}k")
print(f"Difference:        ${abs(my_rmse - sk_rmse):.4f}k")
print(f"Max weight diff:   {np.abs(model.weights - sk.coef_).max():.5f}")

---## 8. Stretch goalsPick one or two. These are where the real learning happens.### A. Add a featureThe dataset has `Latitude` and `Longitude` separately, but house price depends on where you are *jointly*, not on each coordinate alone. Add a feature that captures geographic location better — for example, distance to a famous city like San Francisco (37.77° N, 122.42° W) or Los Angeles (34.05° N, 118.24° W). Does your RMSE drop?### B. Mini-batch gradient descentModify `fit` to use mini-batches of 64 random examples per step instead of the whole training set. Keep the same total number of parameter updates. Does training reach a lower loss faster? What does the loss curve look like now?### C. The normal equation (closed form)For linear regression — and only for linear regression — there's a formula that gives you the optimal weights in one shot, no gradient descent:$$\theta = (X^T X)^{-1} X^T y$$Implement it in three lines. How does its RMSE compare to yours? When would you prefer gradient descent over this closed-form solution?### SubmitSubmit the notebook with all cells run.

#### Stretch A: distance to the coast cities

In [ ]:
# Stretch A: add distance to San Francisco and Los Angeles as features.
def dist_to(lat, lon, lat0, lon0):
    return np.sqrt((lat - lat0) ** 2 + (lon - lon0) ** 2)   # degrees; good enough for a feature

lat, lon = X[:, 6], X[:, 7]
d_sf = dist_to(lat, lon, 37.77, -122.42)
d_la = dist_to(lat, lon, 34.05, -118.24)
d_min = np.minimum(d_sf, d_la)

X_aug = np.column_stack([X, d_sf, d_la, d_min])
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_aug, y, test_size=0.2, random_state=42)
sc = StandardScaler()
Xa_tr_s = sc.fit_transform(Xa_tr)
Xa_te_s = sc.transform(Xa_te)

model_aug = MultivariateLinearRegression(learning_rate=0.1, n_iterations=1000).fit(Xa_tr_s, ya_tr)
aug_rmse = rmse(ya_te, model_aug.predict(Xa_te_s)) * 100
print(f"RMSE with 8 original features:   ${my_rmse:.2f}k")
print(f"RMSE with 3 distance features:   ${aug_rmse:.2f}k")

A straight line in latitude and longitude cannot describe "price falls with distance from a city", but a distance feature can, so the RMSE should drop noticeably.

#### Stretch C: the normal equation

In [ ]:
# Stretch C: closed-form solution theta = (X^T X)^-1 X^T y
Xb = np.column_stack([np.ones(len(X_train_scaled)), X_train_scaled])   # prepend bias column
theta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y_train)                      # solve is safer than inv()

Xb_test = np.column_stack([np.ones(len(X_test_scaled)), X_test_scaled])
ne_rmse = rmse(y_test, Xb_test @ theta) * 100
print(f"Gradient descent RMSE: ${my_rmse:.3f}k")
print(f"Normal equation RMSE:  ${ne_rmse:.3f}k")

The normal equation gives the exact optimum in one shot, so gradient descent should match it almost exactly once it has converged. I would still prefer gradient descent when there are very many features (solving costs roughly O(n³)), when the data is too big to hold in memory, or when `XᵀX` is singular or badly conditioned. It is also the only option for models with no closed form, such as neural networks.